In [1]:
!pip install -q hypothesis

In [2]:
import zlib, itertools, json, random, difflib, re, sys, os, warnings
from hypothesis.errors import Unsatisfiable
from hypothesis import given, settings, strategies as st, assume, HealthCheck, Phase

warnings.filterwarnings("ignore")
HERE = os.getcwd()


In [3]:
PROBLEMS = [
 dict(name="abs_val", args={"x": "int"}, ref=lambda x: abs(x), P="True",
      Q="y >= 0 and (y == x or y == -x)",
      V=[("equiv-rewrite", None, "y == (x if x >= 0 else -x)"),
         ("equiv-reorder", None, "(y == x or y == -x) and y >= 0"),
         ("weak", None, "y >= 0"),
         ("strong", None, "y > 0 and (y == x or y == -x)"),
         ("wrong", None, "y >= 0 and (y == x or y == x + 1)")]),
 dict(name="max2", args={"a": "int", "b": "int"}, ref=lambda a, b: max(a, b), P="True",
      Q="y >= a and y >= b and (y == a or y == b)",
      V=[("equiv-rewrite", None, "y == (a if a >= b else b)"),
         ("weak", None, "y >= a and y >= b"),
         ("weak-2", None, "y >= a and (y == a or y == b)"),
         ("strong", None, "y > a and y > b and (y == a or y == b)")]),
 dict(name="clamp", args={"x": "int", "lo": "int", "hi": "int"}, small=range(-4, 5), ref=lambda x, lo, hi: min(max(x, lo), hi),
      P="lo <= hi",
      Q="lo <= y <= hi and (not (lo <= x <= hi) or y == x) and (not (x < lo) or y == lo) and (not (x > hi) or y == hi)",
      V=[("equiv-rewrite", None, "y == min(max(x, lo), hi)"),
         ("weak", None, "lo <= y <= hi"),
         ("pre-strong", "lo < hi", None),
         ("pre-loose", "lo <= hi + 1", None)]),
 dict(name="is_even", args={"x": "int"}, ref=lambda x: x % 2 == 0, P="True", Q="y == (x % 2 == 0)",
      V=[("equiv-rewrite", None, "(y and x % 2 == 0) or ((not y) and x % 2 != 0)"),
         ("weak", None, "(not y) or x % 2 == 0"),
         ("wrong", None, "y == (x % 4 == 0)")]),
 dict(name="sign", args={"x": "int"}, ref=lambda x: (x > 0) - (x < 0), P="True",
      Q="(x > 0 and y == 1) or (x == 0 and y == 0) or (x < 0 and y == -1)",
      V=[("equiv-rewrite", None, "y in (-1, 0, 1) and ((x > 0) == (y == 1)) and ((x < 0) == (y == -1))"),
         ("weak", None, "(x > 0 and y > 0) or (x == 0 and y == 0) or (x < 0 and y < 0)"),
         ("strong", None, "(x > 0 and y == 1) or (x < 0 and y == -1)"),
         ("wrong", None, "(x >= 0 and y == 1) or (x < 0 and y == -1)")]),
 dict(name="div_floor", args={"a": "int", "b": "int"}, ref=lambda a, b: a // b, P="b > 0",
      Q="y * b <= a and a < y * b + b",
      V=[("equiv-rewrite", None, "y == a // b"),
         ("weak", None, "y * b <= a"),
         ("post-loose", None, "y * b <= a and a <= y * b + b"),
         ("pre-loose", "b >= 0", None),
         ("pre-strong", "b > 1", None)]),
 dict(name="sum_to", args={"n": "nat"}, ref=lambda n: n * (n + 1) // 2, P="n >= 0", Q="y * 2 == n * (n + 1)",
      V=[("equiv-rewrite", None, "y == sum(range(n + 1))"),
         ("weak", None, "y >= 0"),
         ("post-loose", None, "y * 2 >= n * (n + 1)"),
         ("pre-strong", "n > 0", None),
         ("pre-loose", "n >= -1", None)]),
 dict(name="list_max", args={"xs": "list"}, ref=lambda xs: max(xs), P="len(xs) > 0",
      Q="y in xs and all(e <= y for e in xs)",
      V=[("equiv-rewrite", None, "y == max(xs)"),
         ("weak-no-member", None, "all(e <= y for e in xs)"),
         ("weak-no-bound", None, "y in xs"),
         ("strong", None, "y in xs and all(e < y for e in xs)"),
         ("pre-strong", "len(xs) > 1", None),
         ("pre-loose", "len(xs) >= 0", None)]),
 dict(name="count_pos", args={"xs": "list"}, ref=lambda xs: sum(1 for e in xs if e > 0), P="True",
      Q="y == len([e for e in xs if e > 0])",
      V=[("equiv-rewrite", None, "y == sum(1 for e in xs if e > 0)"),
         ("wrong", None, "y == len([e for e in xs if e >= 0])"),
         ("weak", None, "0 <= y <= len(xs)"),
         ("wrong-2", None, "y == len(xs) - len([e for e in xs if e < 0])")]),
 dict(name="remove_idx", args={"xs": "list", "k": "nat"}, ref=lambda xs, k: xs[:k] + xs[k + 1:], P="k < len(xs)",
      Q="len(y) == len(xs) - 1 and all(y[i] == xs[i] for i in range(k)) and all(y[i] == xs[i + 1] for i in range(k, len(y)))",
      V=[("equiv-rewrite", None, "y == xs[:k] + xs[k + 1:]"),
         ("weak-len-only", None, "len(y) == len(xs) - 1"),
         ("weak-no-shift", None, "len(y) == len(xs) - 1 and all(y[i] == xs[i] for i in range(k))"),
         ("pre-strong", "k < len(xs) - 1", None),
         ("pre-loose", "k < len(xs) + 1", None)]),
 dict(name="reverse", args={"xs": "list"}, ref=lambda xs: xs[::-1], P="True",
      Q="len(y) == len(xs) and all(y[i] == xs[len(xs) - 1 - i] for i in range(len(xs)))",
      V=[("equiv-rewrite", None, "y == xs[::-1]"),
         ("weak-len", None, "len(y) == len(xs)"),
         ("weak-perm", None, "sorted(y) == sorted(xs)"),
         ("wrong-identity", None, "len(y) == len(xs) and all(y[i] == xs[i] for i in range(len(xs)))")]),
 dict(name="sort_asc", args={"xs": "list"}, ref=lambda xs: sorted(xs), P="True",
      Q="all(y[i] <= y[i + 1] for i in range(len(y) - 1)) and sorted(y) == sorted(xs)",
      V=[("equiv-rewrite", None, "y == sorted(xs)"),
         ("weak-sorted-only", None, "all(y[i] <= y[i + 1] for i in range(len(y) - 1))"),
         ("weak-perm-only", None, "sorted(y) == sorted(xs)"),
         ("strong-strict", None, "all(y[i] < y[i + 1] for i in range(len(y) - 1)) and sorted(y) == sorted(xs)")]),
 dict(name="contains", args={"xs": "list", "v": "val"}, ref=lambda xs, v: v in xs, P="True", Q="y == (v in xs)",
      V=[("equiv-rewrite", None, "(y and v in xs) or ((not y) and v not in xs)"),
         ("weak", None, "(not y) or v in xs"),
         ("wrong", None, "y == (v in xs and len(xs) > 1)")]),
 dict(name="all_pos", args={"xs": "list"}, ref=lambda xs: all(e > 0 for e in xs), P="True",
      Q="y == all(e > 0 for e in xs)",
      V=[("equiv-rewrite", None, "y == (len([e for e in xs if e <= 0]) == 0)"),
         ("wrong", None, "y == all(e >= 0 for e in xs)"),
         ("weak", None, "(not y) or all(e > 0 for e in xs)")]),
 dict(name="index_of", args={"xs": "list", "v": "val"}, ref=lambda xs, v: xs.index(v), P="v in xs",
      Q="0 <= y < len(xs) and xs[y] == v and all(xs[j] != v for j in range(y))",
      V=[("equiv-rewrite", None, "y == xs.index(v)"),
         ("weak-any-occurrence", None, "0 <= y < len(xs) and xs[y] == v"),
         ("strong-unique", None, "0 <= y < len(xs) and xs[y] == v and all(xs[j] != v for j in range(len(xs)) if j != y)"),
         ("pre-loose", "True", None)]),
]

In [4]:
def mk(expr, argnames, with_y):
    sig = ", ".join(argnames + (["y"] if with_y else []))
    return eval(f"lambda {sig}: ({expr})")


SMALL_LISTS = [list(t) for n in range(4) for t in itertools.product(range(-2, 3), repeat=n)]


def small_domain(t, override=None):
    if t == "int":
        return list(override or range(-8, 9))
    if t == "nat":
        return list(range(0, 5))
    if t == "val":
        return list(range(-2, 3))
    return SMALL_LISTS


def small_inputs(pb):
    doms = [small_domain(t, pb.get("small")) for t in pb["args"].values()]
    return list(itertools.product(*doms))


def small_outputs(sample_y):
    if isinstance(sample_y, bool):
        return [True, False]
    if isinstance(sample_y, int):
        return list(range(-40, 41))
    return SMALL_LISTS


def rand_arg(t, rng):
    if t == "int":
        return rng.randint(-30, 30)
    if t == "nat":
        return rng.randint(0, 8)
    if t == "val":
        return rng.randint(-5, 5)
    return [rng.randint(-5, 5) for _ in range(rng.randint(0, 6))]


def perturb(y, rng):
    if isinstance(y, bool):
        return not y
    if isinstance(y, int):
        return y + rng.choice([-2, -1, 1, 2])
    y = list(y)
    m = rng.randint(0, 4)
    if m == 0 and y:
        y[rng.randrange(len(y))] += rng.choice([-1, 1])
    elif m == 1 and y:
        y.pop(rng.randrange(len(y)))
    elif m == 2:
        y.insert(rng.randint(0, len(y)), rng.randint(-5, 5))
    elif m == 3:
        y.reverse()
    else:
        y = [rng.randint(-5, 5) for _ in range(rng.randint(0, 6))]
    return y

In [6]:
REL = ["P_sound", "P_complete", "Q_sound", "Q_complete"]


def oracle(pb, P, Q, Ph, Qh):
    res = dict.fromkeys(REL, True)
    for x in small_inputs(pb):
        p, ph = P(*x), Ph(*x)
        if p and not ph:
            res["P_sound"] = False
        if ph and not p:
            res["P_complete"] = False
        if not p:
            continue
        ys = small_outputs(pb["ref"](*x))
        for y in ys:
            q, qh = Q(*x, y), Qh(*x, y)
            if qh and not q:
                res["Q_sound"] = False
            if q and not qh:
                res["Q_complete"] = False
        if all(not v for v in res.values()):
            break
    return res

In [7]:
def make_tests(pb, P, Q, rng):
    names = list(pb["args"])
    pos, negpost, negpre = [], [], []
    tries = 0
    while len(pos) < 5 and tries < 2000:
        tries += 1
        x = tuple(rand_arg(t, rng) for t in pb["args"].values())
        if P(*x):
            pos.append((x, pb["ref"](*x)))
    for x, y in pos:
        got = 0
        for _ in range(200):
            y2 = perturb(y, rng)
            if y2 != y and not Q(*x, y2):
                negpost.append((x, y2)); got += 1
            if got >= 3:
                break
    for _ in range(2000):
        x = tuple(rand_arg(t, rng) for t in pb["args"].values())
        if not P(*x):
            negpre.append(x)
        if len(negpre) >= 3:
            break
    return pos, negpost, negpre


def eval_fixed(tests, Ph, Qh):
    pos, negpost, negpre = tests
    return dict(
        P_sound=all(Ph(*x) for x, _ in pos),
        P_complete=all(not Ph(*x) for x in negpre),
        Q_sound=all(not Qh(*x, y) for x, y in negpost),
        Q_complete=all(Qh(*x, y) for x, y in pos))



def xy_strategy(pb):
    ts = list(pb["args"].values())

    def arg(t):
        return {"int": st.integers(-30, 30), "nat": st.integers(0, 8), "val": st.integers(-5, 5),
                "list": st.lists(st.integers(-5, 5), max_size=6)}[t]
    return st.tuples(*[arg(t) for t in ts]), st.randoms(use_true_random=False), st.integers(0, 2)


def eval_pbt(pb, P, Q, Ph, Qh, max_examples=200):
    xs_s, rng_s, mode_s = xy_strategy(pb)
    out = {}
    settings_kw = dict(max_examples=max_examples, deadline=None, derandomize=True,
                       suppress_health_check=list(HealthCheck), phases=[Phase.generate])

    def find_counterexample(prop):
        try:
            @settings(**settings_kw)
            @given(xs_s, rng_s, mode_s)
            def t(x, rng, mode):
                prop(x, rng, mode)
            t()
            return True
        except AssertionError:
            return False
        except Unsatisfiable:
            return True

    def gen_y(x, rng, mode):
        y = pb["ref"](*x)
        if mode == 1:
            return perturb(y, rng)
        if mode == 2:
            return perturb(perturb(y, rng), rng)
        return y

    def p_sound(x, rng, mode):
        assume(P(*x)); assert Ph(*x)

    def p_complete(x, rng, mode):
        assume(Ph(*x)); assert P(*x)

    def q_sound(x, rng, mode):
        assume(P(*x)); y = gen_y(x, rng, mode); assume(Qh(*x, y)); assert Q(*x, y)

    def q_complete(x, rng, mode):
        assume(P(*x)); y = gen_y(x, rng, mode); assume(Q(*x, y)); assert Qh(*x, y)

    return dict(P_sound=find_counterexample(p_sound), P_complete=find_counterexample(p_complete),
                Q_sound=find_counterexample(q_sound), Q_complete=find_counterexample(q_complete))


def tok(s):
    return re.findall(r"[A-Za-z_]+|\d+|==|<=|>=|!=|[^\sA-Za-z_\d]", s)


def text_sim(pb, Ph_s, Qh_s):
    ref = tok(pb["P"] + " ## " + pb["Q"])
    cand = tok(Ph_s + " ## " + Qh_s)
    return difflib.SequenceMatcher(None, ref, cand).ratio()


In [8]:
rows = []
rng = random.Random(2026)
for pb in PROBLEMS:
    names = list(pb["args"])
    P, Q = mk(pb["P"], names, False), mk(pb["Q"], names, True)
    tests = make_tests(pb, P, Q, random.Random(zlib.crc32(pb["name"].encode()) % 10**6 + 7))
    cands = [("ground-truth-copy", pb["P"], pb["Q"])] + [(l, p or pb["P"], q or pb["Q"]) for l, p, q in pb["V"]]
    for label, ps, qs in cands:
        Ph, Qh = mk(ps, names, False), mk(qs, names, True)
        o = oracle(pb, P, Q, Ph, Qh)
        a = eval_fixed(tests, Ph, Qh)
        b = eval_pbt(pb, P, Q, Ph, Qh)
        rows.append(dict(problem=pb["name"], label=label, P=ps, Q=qs, oracle=o, fixed=a, pbt=b,
                         sim=text_sim(pb, ps, qs), n_pos=len(tests[0]), n_neg=len(tests[1]) + len(tests[2])))
json.dump(rows, open(os.path.join(HERE, "tn3_rows.json"), "w"), indent=1, ensure_ascii=False)

In [9]:
n = len(rows)
good = [r for r in rows if all(r["oracle"].values())]
print(f"problems={len(PROBLEMS)} candidates={n} fully-correct(oracle)={len(good)} flawed={n-len(good)}")
print("flaw type counts (oracle):", {k: sum(not r['oracle'][k] for r in rows) for k in REL})

print("\nRelation-level detection of REAL flaws (recall) and false alarms")
print(f"{'relation':12s} {'real flaws':>10s} {'EVAL-A caught':>14s} {'EVAL-B caught':>14s} {'A false alarm':>14s} {'B false alarm':>14s}")
tot = dict(real=0, a=0, b=0)
for k in REL:
    real = [r for r in rows if not r["oracle"][k]]
    ca = sum(not r["fixed"][k] for r in real)
    cb = sum(not r["pbt"][k] for r in real)
    ok = [r for r in rows if r["oracle"][k]]
    fa = sum(not r["fixed"][k] for r in ok)
    fb = sum(not r["pbt"][k] for r in ok)
    tot["real"] += len(real); tot["a"] += ca; tot["b"] += cb
    print(f"{k:12s} {len(real):10d} {ca:14d} {cb:14d} {fa:14d} {fb:14d}")
print(f"{'TOTAL':12s} {tot['real']:10d} {tot['a']:14d} {tot['b']:14d}")
print(f"recall A = {100*tot['a']/tot['real']:.1f}%   recall B = {100*tot['b']/tot['real']:.1f}%")

for name, key in (("EVAL-A fixed tests", "fixed"), ("EVAL-B property-based", "pbt")):
    acc = [r for r in rows if all(r[key].values())]
    fp = [r for r in acc if not all(r["oracle"].values())]
    rej = [r for r in rows if not all(r[key].values())]
    fn = [r for r in rej if all(r["oracle"].values())]
    corr = sum(all(r[key].values()) == all(r["oracle"].values()) for r in rows)
    print(f"{name:24s} candidate-level accuracy={corr}/{n} ({100*corr/n:.1f}%); flawed-but-accepted={len(fp)}; correct-but-rejected={len(fn)}")
    if key == "fixed":
        print("   examples flawed-but-accepted:", [(r['problem'], r['label']) for r in fp][:12])
    else:
        print("   examples flawed-but-accepted:", [(r['problem'], r['label']) for r in fp][:12])


def auc(pos, neg):
    return sum((p > q) + 0.5 * (p == q) for p in pos for q in neg) / (len(pos) * len(neg))
flawed = [r for r in rows if not all(r["oracle"].values())]
print(f"\nTEXT similarity: mean(correct)={sum(r['sim'] for r in good)/len(good):.3f}  mean(flawed)={sum(r['sim'] for r in flawed)/len(flawed):.3f}  AUC={auc([r['sim'] for r in good],[r['sim'] for r in flawed]):.3f}")
eq = [r for r in rows if r["label"].startswith("equiv")]
print(f"equivalent rewrites: {len(eq)}; all correct per oracle: {all(all(r['oracle'].values()) for r in eq)}; mean text sim={sum(r['sim'] for r in eq)/len(eq):.3f}")
print("disagreements EVAL-B vs oracle (B says pass but oracle fails or vice versa):",
      [(r["problem"], r["label"], k) for r in rows for k in REL if r["pbt"][k] != r["oracle"][k]][:20])

problems=15 candidates=78 fully-correct(oracle)=32 flawed=46
flaw type counts (oracle): {'P_sound': 5, 'P_complete': 5, 'Q_sound': 30, 'Q_complete': 14}

Relation-level detection of REAL flaws (recall) and false alarms
relation     real flaws  EVAL-A caught  EVAL-B caught  A false alarm  B false alarm
P_sound               5              1              5              0              0
P_complete            5              3              5              0              0
Q_sound              30             24             29              0              0
Q_complete           14              9             14              0              0
TOTAL                54             37             53
recall A = 68.5%   recall B = 98.1%
EVAL-A fixed tests       candidate-level accuracy=64/78 (82.1%); flawed-but-accepted=14; correct-but-rejected=0
   examples flawed-but-accepted: [('abs_val', 'strong'), ('max2', 'weak-2'), ('clamp', 'pre-strong'), ('clamp', 'pre-loose'), ('sign', 'strong'), ('sign', 'wro